# Bronze Layer Ingestion Pipeline - Generic & Metadata-Driven

Generic notebook to ingest ANY source into Bronze Delta tables using LakeForge framework configuration.

**Supported input formats:** csv, tsv, psv, json, jsonl/ndjson, parquet, avro, orc, xml, xlsx/xls (Excel), delta, and plain/fixed-width text — selected in config or auto-detected from the file extension via the UniversalLoader.

In [0]:
%pip install openpyxl xlrd chardet


In [0]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
dbutils.widgets.text("source_config_file", "customers", "Source Config File / Usecase Name")
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "staging", "prod"], "Environment / Stage")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.dropdown("create_views", "false", ["false", "true"], "Create Views (Optional)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

# Workspace file access from cluster compute requires the /Workspace prefix
if not config_dir.startswith("/Workspace") and os.path.isdir("/Workspace" + config_dir):
    config_dir = "/Workspace" + config_dir
if not repo_root.startswith("/Workspace") and os.path.isdir("/Workspace" + repo_root):
    repo_root = "/Workspace" + repo_root

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

source_config_file = dbutils.widgets.get("source_config_file")
try:
    _stage = dbutils.widgets.get("stage").strip()
except Exception:
    _stage = ""
environment = _stage or dbutils.widgets.get("environment")
client_id = dbutils.widgets.get("client_id").strip() or None
create_views = dbutils.widgets.get("create_views").lower() == "true"

print(f"Executing Bronze Ingestion for config: '{source_config_file}' in {environment} (client: {client_id or 'global'})")


In [0]:
# Imports & Path setup
import sys

from dataclasses import asdict
from lakeforge import (
    ConfigParser,
    create_dq_engine,
    create_bronze_writer,
    create_trust_engine
)

from pyspark.sql import functions as F
from datetime import datetime

print("✅ Framework libraries loaded")

In [0]:
# Load Configurations - resolve ALL sources to run
# source_config_file can be:
#   - a usecase FILE name (e.g. "bad_duplicate_keys" -> every source inside
#     configs/ingestion/bad_duplicate_keys.json runs)
#   - a single source config name (e.g. "customers" -> only that source runs)
# client_id (if set) runs only sources whose config carries that client_id.
ingestion_sources = ConfigParser.get_ingestion_sources(config_dir, source_config_file, environment, client_id=client_id)

print(f"Resolved {len(ingestion_sources)} source(s) for '{source_config_file}'"
      + (f" (client: {client_id})" if client_id else " (all clients)"))
for ing in ingestion_sources:
    print(f"  - {ing.source_name} -> {ing.target_catalog}.{ing.target_schema}.{ing.target_table} [{ing.source_type}]")


In [0]:
# Per-source Bronze ingestion: load -> DQ -> segregate -> write -> trust score
# A bronze table that does not exist yet is created automatically by the write.
from lakeforge.ingestion import create_universal_loader

universal_loader = create_universal_loader(spark)


def run_bronze_ingestion(ing_config):
    """Run the complete bronze flow for one source and return a summary dict."""
    source = ing_config.source_name
    target_table = f"{ing_config.target_catalog}.{ing_config.target_schema}.{ing_config.target_table}"
    print(f"\n{'='*70}\n▶ Source: {source}")

    # 1. Load raw data (any format - from config or auto-detected from the extension)
    # Incremental sources ("incremental": true in config) load ONLY files not
    # yet ingested - tracked in the file-inventory table after each successful write.
    inventory = None
    load_files = None
    if getattr(ing_config, "incremental", False):
        from lakeforge.ingestion.incremental import FileInventoryManager
        log_table = (
            getattr(ing_config, "incremental_log_table", None)
            or f"{ing_config.target_catalog}.monitoring.ingested_files"
        )
        inventory = FileInventoryManager(spark, log_table)
        all_files = universal_loader.list_source_files(
            ing_config.source_path, ing_config.file_format or ing_config.source_type
        )
        load_files = inventory.list_new_files(ing_config.source_name, all_files)
        if not load_files:
            print(f"⏭️ No new files for '{source}' - all {len(all_files)} file(s) already ingested. Skipping.")
            return {
                "source": source,
                "table": target_table,
                "created": False,
                "rows_loaded": 0,
                "rows_written": 0,
                "quarantined": 0,
                "trust_score": 100.0,
                "trust_level": "SKIPPED (no new files)",
            }
        print(f"📦 Incremental: loading {len(load_files)} of {len(all_files)} file(s) (new only)")

    df_raw = universal_loader.load_from_config(ing_config, files=load_files)
    raw_count = df_raw.count()
    print(f"✅ Loaded {raw_count} rows from {ing_config.source_path} [{ing_config.file_format or ing_config.source_type}]")

    # 2. DQ validation (rules from configs/dq/<source>.json - optional, may be empty)
    dq_config = ConfigParser.get_dq_config(config_dir, source, environment, client_id=client_id)
    rules_list = [asdict(r) for r in dq_config.rules]
    dq_engine = create_dq_engine(spark)
    dq_results = dq_engine.validate_dataframe(df=df_raw, rules=rules_list, quarantine_failures=True)
    print(f"🔍 DQ: {dq_results['rules_passed']}/{dq_results['rules_executed']} rules passed")

    # 3. Segregate clean vs quarantined records
    business_keys = ing_config.business_key
    if dq_results["quarantine_count"] > 0 and dq_results["quarantine_df"] is not None:
        if business_keys:
            clean_df = df_raw.join(dq_results["quarantine_df"], on=business_keys, how="left_anti")
        else:
            clean_df = df_raw.subtract(dq_results["quarantine_df"])
    else:
        clean_df = df_raw

    # 4. Write clean records (creates the bronze table automatically if missing)
    table_existed = spark.catalog.tableExists(target_table)
    bronze_writer = create_bronze_writer(spark)
    bronze_writer.write_to_bronze(
        df=clean_df,
        target_table=ing_config.target_table,
        catalog=ing_config.target_catalog,
        schema=ing_config.target_schema,
        mode=ing_config.mode,
        merge_keys=business_keys,
        add_audit_columns=True,
        source_system=source,
        schema_evolution=ing_config.schema_evolution
    )
    print(f"✅ {'Appended to existing table' if table_existed else 'Created new table'}: {target_table} ({clean_df.count()} rows)")

    # 4b. Record the ingested files so the next incremental run skips them
    if inventory is not None:
        inventory.record_files(ing_config.source_name, load_files)
        print(f"📦 Recorded {len(load_files)} file(s) in inventory table {inventory.table}")

    # 5. Quarantine failed records (if any)
    if dq_results["quarantine_count"] > 0:
        dq_engine.write_quarantine_table(
            quarantine_df=dq_results["quarantine_df"],
            catalog=dq_config.catalog,
            schema=dq_config.schema,
            table_name=dq_config.quarantine_table,
            validation_results=dq_results
        )
        print(f"⚠️ Quarantined {dq_results['quarantine_count']} rows in {dq_config.quarantine_table}")

    # 6. Trust score
    trust_engine = create_trust_engine(spark)
    trust_score = trust_engine.calculate_trust_score(
        table_name=target_table,
        dq_results=dq_results,
        pipeline_stage="bronze"
    )
    print(f"🎯 Trust Score: {trust_score['overall_score']:.1f}% ({trust_score['trust_level']})")

    return {
        "source": source,
        "table": target_table,
        "created": not table_existed,
        "rows_loaded": raw_count,
        "rows_written": clean_df.count(),
        "quarantined": dq_results["quarantine_count"],
        "trust_score": trust_score["overall_score"],
        "trust_level": trust_score["trust_level"],
    }

In [0]:
# Run Bronze ingestion for EVERY resolved source (table created if missing)
ingestion_results = [run_bronze_ingestion(ing) for ing in ingestion_sources]

print(f"\n{'='*70}")
print(f"🏁 BRONZE INGESTION COMPLETE - {len(ingestion_results)} source(s) processed")
for r in ingestion_results:
    print(f"  {r['source']}: {'CREATED' if r['created'] else 'appended'} {r['table']} "
          f"| loaded {r['rows_loaded']}, wrote {r['rows_written']}, quarantined {r['quarantined']} "
          f"| trust {r['trust_score']:.1f}% ({r['trust_level']})")

In [0]:
# Create Bronze-layer views if explicitly enabled (otherwise run VIEWS_Creation_Pipeline on-demand)
# Views are read from configs/views/bronze/*.json - add bronze views there as JSON files.
# client_id set -> only views whose 'client_id' matches; blank client -> all bronze views.
# Views whose source tables do not exist yet are skipped individually with the reason shown.
if create_views:
    try:
        print("Checking for configured views to create...")
        # Reload so parser edits made during a live session are picked up
        import importlib
        import lakeforge.metadata.config_parser as _cfgmod
        importlib.reload(_cfgmod)
        from lakeforge.views.view_manager import create_view_manager
        from lakeforge.metadata.config_parser import load_view_configs, ConfigParser
        env_config = ConfigParser.get_environment_config(config_dir, environment)
        # Bronze-layer views only: configs/views/bronze/*.json
        target_views = load_view_configs(config_dir, environment, client_id=client_id, layer="bronze")

        if target_views:
            view_manager = create_view_manager(spark, env_config)
            created, skipped = [], []
            for vc in target_views:
                try:
                    view_manager.create_view(vc)
                    created.append(vc.view_name)
                except Exception as e:
                    reason = str(e).strip().splitlines()[0][:120]
                    skipped.append(f"{vc.view_name} -> {reason}")
            if created:
                print(f"✅ Created {len(created)} view(s): " + ", ".join(created))
            if skipped:
                print(f"⚠️ Skipped {len(skipped)} view(s) (source table missing or error):")
                for s in skipped:
                    print(f"   - {s}")
            if not created and not skipped:
                print("No views configured for this run's scope")
        else:
            print("No views configured for this run's scope")
    except Exception as e:
        print(f"Error creating views: {str(e)}")
else:
    print("View creation skipped (set create_views=true or run VIEWS_Creation_Pipeline on demand)")